# Converting cubed sphere to lat/lon grid with JEDI

## Prerequisite 

Please review [here](https://mer-a-o.github.io/howtojedi/discover/) to learn about setting up your work environment and configuring the `jedi-bundle`. 

Before starting, verify:
- [ ] `$JEDI_BUILD` environment variable is set
- [ ] `fv3jedi_converttostructuredgrid.x` exists in `$JEDI_BUILD`
- [ ] Input files exist in the `jedi_utils/run_convert_to_latlon/inputs` directory (background, observation, and auxiliary files such as geometry configurations)
- [ ] You have access to compute resources (interactive node or batch system)

---

## Usage and limitations
`fv3jedi_converttostructuredgrid.x` can be used to convert a cubed sphere file to a structured (lat/lon) grid. However, the user needs to specify which model level they want to perform the conversion on. This means that the executable doesn't automatically regrid all the model levels.

An example of an input YAML file is below.
You will need to specify:
- Input geometry (under `state geometry`)
- Details about the input file (under `state`)
- Details about the `structured grid interpolation`. More details available [here].(https://github.com/JCSDA/oops/blob/1.12.0/src/oops/base/StructuredGridWriter.h#L292-L344)

### YAML structure:

```yaml
states to structured grid:
- state geometry:
    fms initialization:
      namelist filename: geometry_input/fmsmpp.nml
      field table filename: geometry_input/field_table_gmao
    akbk: geometry_input/akbk72.nc4
    npx: 13
    npy: 13
    npz: 72
    field metadata override: geometry_input/geos_cf.yaml

  state:
    datetime: 2020-09-03T15:00:00Z
    filetype: cube sphere history
    state variables: [DELP,NO2,NO,O3,PS,SPHU]
    datapath: geos_c12/
    filename: geos_cf.bkg.20200903_150000z.nc4
    state variables: &statevars
    - air_pressure_thickness
    - volume_mixing_ratio_of_no2
    - volume_mixing_ratio_of_no
    - volume_mixing_ratio_of_o3
    - air_pressure_at_surface
    - water_vapor_mixing_ratio_wrt_moist_air
    field io names: &field_io_names
      air_pressure_thickness: DELP
      volume_mixing_ratio_of_no2: NO2
      volume_mixing_ratio_of_no: "NO"
      volume_mixing_ratio_of_o3: O3
      air_pressure_at_surface: PS
      water_vapor_mixing_ratio_wrt_moist_air: SPHU


  structured grid interpolation:
    grid type: regular gaussian
    local interpolator type: atlas interpolator
    interpolation method:
      type: finite-element
    number of latitude gridpoints: 12
    variables to output: *statevars
    pressure levels in hPa:
    - 925
    - 850
    - 700
    - 500
    - 250
    all model levels: true
    bottom model level: false
    datapath: outputs
    prefix: converttogauss
```

### Running convertstate

```bash
$MPIEXEC "-n" "6" $JEDI_BUILD/fv3jedi_converttostructuredgrid.x converttolatlon_geos.yaml 2>&1 | tee log_convert_to_latlon.txt
```

In this example, two files will be generated. The "PressureLevels" file will include the `variables to output` values in latlon grid for levels specified under `pressure levels in hPa`. Since, `all model levels` is true the "modelLevels" files will include the `variables to output` values in latlon grid for all model levels. To learn more about other options refer to `StructuredGridWriter.h` in [OOPS](https://github.com/JCSDA/oops/blob/1.12.0/src/oops/base/StructuredGridWriter.h#L292-L344).

---

## Ctest example:

An example of this application is ctest `fv3jedi_test_tier1_converttolatlon_gfs` in [fv3-jedi](https://github.com/JCSDA/fv3-jedi/blob/develop/test/CMakeLists.txt#L1050-L1053). In this [example](https://github.com/JCSDA/fv3-jedi/blob/develop/test/testinput/converttogauss_gfs.yaml), filetype of `fms restart` is converted from cubed sphere to latlon grid. 